<a href="https://colab.research.google.com/github/type3005/-/blob/main/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ส่วนที่ 0 — Import

In [69]:
import random
import time
from datetime import datetime
!pip install Faker
from faker import Faker
fake = Faker("th_TH")
random.seed(1)
import pandas as pd
import matplotlib, os, shutil
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

import random
from datetime import datetime, timedelta

In [70]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

# 5. สร้างข้อมูลจำลองและแสดงผล
data = {
    'เมือง': ['กรุงเทพฯ', 'เชียงใหม่', 'ภูเก็ต'],
    'อุณหภูมิ': [30.5, 25.2, 28.9]
}
df_thai = pd.DataFrame(data)

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-thai-tlwg is already the newest version (1:0.7.3-1).
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.
ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!


## ส่วนที่ 1 — เตรียม class และฟังก์ชัน

In [71]:
class Member:
    """ข้อมูลสมาชิกธนาคารออมทรัพย์"""
    def __init__(self, member_id, customer_name, citizen_id="-", phone_number="-"):
        self.member_id = member_id
        self.customer_name = customer_name
        self.citizen_id = citizen_id
        self.phone_number = phone_number

    def get_info(self):
        return f"ลูกค้า ID: {self.member_id} | ชื่อ: {self.customer_name} | เลขบัตรประชาชน: {self.citizen_id} | เบอร์โทร: {self.phone_number}"

In [72]:
class Account:
    """บัญชีเงินฝากธนาคารออมทรัพย์"""
    def __init__(self, account_number, balance, owner, interest_rate=0.015):
        self.account_number = account_number
        self.balance = float(balance)
        self.owner = owner #ชื่อเจ้าของบัญชี
        self.interest_rate = interest_rate  # ดอกเบี้ย 1.5% ต่อปี ( default )

    # Validation & Operations
    def deposit(self, amount):
        """ฝากเงิน: balance = balance + amount"""
        self.balance += amount
        return "ฝากเงินสำเร็จ"

    def withdraw(self, amount):
        """ถอนเงิน: ตรวจสอบ balance >= amount"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอ (มีอยู่ {self.balance:,.2f} บาท)"
        self.balance -= amount
        return "ถอนเงินสำเร็จ"

    def transfer(self, target_account, amount):
        """โอนเงิน: ตัดบัญชีต้นทาง และบวกเข้าบัญชีปลายทาง"""
        if self.balance < amount:
            return f" ยอดเงินไม่พอโอน (มีอยู่ {self.balance:,.2f} บาท)"
        self.balance -= amount
        target_account.balance += amount
        return "โอนเงินสำเร็จ"

    def apply_interest(self):
        """คำนวณดอกเบี้ย: interest = balance * interest_rate แล้วบวกเข้ายอดคงเหลือ"""
        interest = self.balance * self.interest_rate
        self.balance += interest
        return interest

In [91]:
class Transaction:
    """Class บันทึกประวัติสลิปรายการธุรกรรมธนาคาร"""
    def __init__(self, txn_id, account, transaction_type, amount, target_account=None):
        self.txn_id = txn_id
        self.queue_number = f"A-{txn_id:03d}"


        # เก็บ Object Account เพื่อนำไปคิดเงินและดอกเบี้ยต่อ
        self.account = account
        self.account_number = account.account_number

        # ดึงชื่อลูกค้าทะลุจาก Account -> Member
        self.customer_name = account.owner.customer_name

        self.transaction_type = transaction_type
        self.amount = amount
        self.target_account = target_account  # บัญชีปลายทาง (ถ้าเป็นโอนเงิน)


        # 📌 โค้ดสุ่มวันที่และเวลา (08:00 - 16:00 น.) อยู่ตรงนี้
        start_date = datetime(2026, 1, 1)
        end_date = datetime(2026, 8, 22)
        random_days = random.randint(0, (end_date - start_date).days)
        random_date = start_date + timedelta(days=random_days)

        random_seconds = random.randint(0, 8 * 3600)
        dt = datetime(
            random_date.year, random_date.month, random_date.day, 8, 0, 0
        ) + timedelta(seconds=random_seconds)

        self.txn_date = dt.strftime("%d/%m/%Y")
        self.time = dt.strftime("%H:%M:%S")

    def to_dict(self):
        interest = getattr(self.account, 'yearly_interest', 0.0)
        return {
            "ID รายการ": self.txn_id,
            "หมายเลขคิว": self.queue_number,
            "เลขบัญชี": self.account_number,
            "ชื่อลูกค้า": self.customer_name,
            "ประเภทรายการ": self.transaction_type,
            "จำนวนเงิน": self.amount,
            "บัญชีปลายทาง": self.target_account if self.target_account else "-",
            "ยอดหลังทำรายการ": self.account.balance,
            "ดอกเบี้ยสิ้นปี (1.5%)": interest,                         # 📌 เพิ่มคอลัมน์ดอกเบี้ย
            "ยอดรวมดอกเบี้ยสุทธิ": self.account.balance + interest,    # 📌 เพิ่มคอลัมน์ยอดรวมสุทธิ
           "วันที่ทำรายการ": self.txn_date,  # 📌 เพิ่มคอลัมน์วันที่
            "เวลาทำรายการ": self.time
        }

In [74]:
def generate_thai_name():
    """ฟังก์ชัน: สุ่มชื่อและนามสกุลลูกค้าแยกกัน"""
    name = fake.name()
    first_name, last_name = name.split(" ", 1)

    return f"{first_name} {last_name}"

def random_amount(min_val=100.0, max_val=2000.0):
    """ฟังก์ชัน: สุ่มยอดเงิน -> คืนค่าเป็น float """
    return round(random.uniform(min_val, max_val), 2)

def format_currency(amount, symbol="บาท"):
    """ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตริงราคา -> คืนค่าเป็น string"""
    return f"{amount:,.2f} {symbol}"

## ส่วนที่ 2 — ทดสอบฟังก์ชันทีละตัว ก่อนเอาไปประกอบเป็นกระบวนการ

In [75]:
# เรียก generate_thai_name() 3 ครั้ง -> ทุกครั้งได้ชื่อสุ่มไม่ซ้ำแบบ (แสดงว่าฟังก์ชันทำงานทุกครั้งที่เรียก)
for _ in range(3):
    print("ชื่อที่สุ่มได้:", generate_thai_name())

ชื่อที่สุ่มได้: วนาลี แนวพญา
ชื่อที่สุ่มได้: เถลิงยศ ธาราธร
ชื่อที่สุ่มได้: ประวี ทับทิมไทย


In [76]:
print("\n# เรียก random_amount() 3 ครั้ง")
for _ in range(3):
    print("ยอดเงินสุ่มได้:", format_currency(random_amount()))


# เรียก random_amount() 3 ครั้ง
ยอดเงินสุ่มได้: 355.29 บาท
ยอดเงินสุ่มได้: 1,710.12 บาท
ยอดเงินสุ่มได้: 1,551.17 บาท


## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา (ให้เห็นว่าฟังก์ชันเรียกฟังก์ชัน/method อื่นต่อได้)

In [77]:
def explain_transaction_calculation(transaction):
    """ฟังก์ชันคำนวณเงิน ฝาก/ถอน/โอน และเรียกใช้ Method ของ Account"""

    account = transaction.account
    amount = float(transaction.amount)
    txn_type = transaction.transaction_type

    print(f"หมายเลขคิว = '{transaction.queue_number}'")
    print(f"หมายเลขบัญชี = '{account.account_number}'")
    print(f"ชื่อลูกค้า = '{transaction.customer_name}'")
    print(f"ประเภทรายการ = '{txn_type}'")
    print(f"ยอดเงินก่อนทำรายการ = {format_currency(account.balance)}")

    # 📌 เรียกใช้ Method ภายใน Class Account
    if txn_type == "ฝากเงิน":
        status_msg = account.deposit(amount)

    elif txn_type == "ถอนเงิน":
        status_msg = account.withdraw(amount)
        # ❌ ถ้าเงินไม่พอ ให้หยุดประมวลผลทันที
        if "ยอดเงินไม่พอ" in status_msg:
            print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
            print(f"สถานะรายการ = '{status_msg}'")
            return False

    elif txn_type == "โอนเงิน":
        if transaction.target_account:
            print(f"บัญชีปลายทาง = '{transaction.target_account}'")

        dummy_member = Member(0, "บัญชีปลายทาง")
        dummy_target = Account("987-6-00000-0", balance=0.0, owner=dummy_member)
        status_msg = account.transfer(dummy_target, amount)
        # ❌ ถ้าเงินไม่พอโอน ให้หยุดประมวลผลทันที
        if "ยอดเงินไม่พอ" in status_msg:
            print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
            print(f"สถานะรายการ = '{status_msg}'")
            return False

    # คำนวณดอกเบี้ย (จะทำเฉพาะรายการที่สำเร็จเท่านั้น)
    interest_val = account.apply_interest()

    print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
    print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
    print(f"สถานะรายการ = '{status_msg}'")
    print(f"ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = {format_currency(interest_val)}")

    return True

In [78]:
import random

# 1. กำหนด seed และเตรียมตัวแปร
random.seed(1)
transactions = []

# 2. Loop สุ่มข้อมูล 300 รายการ
for i in range(1, 301):
    name = generate_thai_name()
    amount = random_amount()
    service = random.choice(["ฝากเงิน", "ถอนเงิน", "โอนเงิน"])

    member = Member(member_id=1 + i, customer_name=name)

    initial_balance = round(random.uniform(100, 5000), 2)

    # 🎲 สุ่มเลขบัญชีลูกค้า
    acc_p1 = random.randint(100, 999)
    acc_p2 = random.randint(1, 9)
    acc_p3 = random.randint(10000, 99999)
    random_account_no = f"{acc_p1}-{acc_p2}-{acc_p3:05d}-0"

    account = Account(account_number=random_account_no, balance=initial_balance, owner=member)
    target_acc = f"987-6-{random.randint(10000, 99999)}-0" if service == "โอนเงิน" else None
    # 📌 คำนวณยอดเงินผ่าน Method ของ Account โดยตรง
    if service == "ฝากเงิน":
        account.deposit(amount)
    elif service == "ถอนเงิน":
        account.withdraw(amount)
    elif service == "โอนเงิน":
        dummy_mem = Member(0, "ปลายทาง")
        dummy_acc = Account("987-6-00000-0", balance=0.0, owner=dummy_mem)
        account.transfer(dummy_acc, amount)

    # ประมวลผลดอกเบี้ย
    account.apply_interest()

    transaction = Transaction(
        txn_id=i,
        account=account,
        transaction_type=service,
        amount=amount,
        target_account=target_acc
    )

    transactions.append(transaction)

# 3. แสดงตัวอย่างรายการแรก (คิว A-001)
print("\n--- [ตัวอย่างการแสดงผลรายการแรก (คิว A-001)] ---")
explain_transaction_calculation(transactions[0])


--- [ตัวอย่างการแสดงผลรายการแรก (คิว A-001)] ---
หมายเลขคิว = 'A-001'
หมายเลขบัญชี = '607-8-71898-0'
ชื่อลูกค้า = 'พาสุข ถนัดหัตถกรรม'
ประเภทรายการ = 'ฝากเงิน'
ยอดเงินก่อนทำรายการ = 1,730.71 บาท
จำนวนเงินทำรายการ = 355.29 บาท
ยอดเงินคงเหลือหลังทำรายการ = 2,117.29 บาท
สถานะรายการ = 'ฝากเงินสำเร็จ'
ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = 31.29 บาท


True

## ส่วนที่ 4 — จำลอง "ลูกค้า 1 คนเดินเข้าร้าน" แบบ step-by-step

In [79]:
import random
import time

def simulate_customer_visit(txn_id, customer_name, pause=0.5):
    """จำลองขั้นตอนลูกค้า 1 คนเดินเข้าธนาคาร"""
    print("=" * 60)
    queue_no = f"A-{txn_id:03d}"
    print(f"🎫 [ผู้ออกบัตรคิว] คุณ '{customer_name}' กดรับบัตรคิว ได้หมายเลข: {queue_no}")

    # 1. สร้าง Member
    member = Member(member_id=100 + txn_id, customer_name=customer_name)

    # 2. 🎲 สุ่มยอดเงินตั้งต้น (1,000 - 5,000 บาท)
    initial_balance = round(random.uniform(100, 5000), 2)

    # 3. 🎲 สุ่มเลขบัญชีลูกค้า (รูปแบบ 123-4-56789-0)
    acc_p1 = random.randint(100, 999)
    acc_p2 = random.randint(1, 9)
    acc_p3 = random.randint(10000, 99999)
    random_account_no = f"{acc_p1}-{acc_p2}-{acc_p3:05d}-0"

    # 4. สร้าง Account
    account = Account(account_number=random_account_no, balance=initial_balance, owner=member)

    # 5. สุ่มประเภทรายการ และจำนวนเงิน
    service = random.choice(["ฝากเงิน", "ถอนเงิน", "โอนเงิน"])
    amount = random_amount()
    target_acc = f"987-6-{random.randint(10000, 99999)}-0" if service == "โอนเงิน" else None

    # 6. ประมวลผลธุรกรรม
    print(f"🔔 [เชิญหมายเลข {queue_no}] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: '{service}'")
    time.sleep(pause)
    print("⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)")
    time.sleep(pause)

    if service == "ฝากเงิน":
        account.deposit(amount)
    elif service == "ถอนเงิน":
        account.withdraw(amount)
    elif service == "โอนเงิน":
        dummy_mem = Member(0, "ปลายทาง")
        dummy_acc = Account("987-6-00000-0", balance=0.0, owner=dummy_mem)
        account.transfer(dummy_acc, amount)

    # คิดดอกเบี้ย
    account.apply_interest()

    # 7. สร้าง Transaction
    txn = Transaction(
        txn_id=txn_id,
        account=account,
        transaction_type=service,
        amount=amount,
        target_account=target_acc
    )

    # แสดงรายละเอียดคำนวณ
    print("💻 เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:")
    explain_transaction_calculation(txn)

    # 📌 เพิ่มส่วนพิมพ์สลิปตรงนี้ครับ
    print("✅ ทำรายการสำเร็จ!")
    timestamp_str = time.strftime("%H:%M:%S")
    # ปรับรูปแบบชื่อกรณีที่เป็น Tuple/List ให้แสดงเป็นข้อความเรียบง่าย
    display_name = " ".join(customer_name) if isinstance(customer_name, (tuple, list)) else customer_name
    print(f"🧾 สลิปบันทึกรายการ #{txn_id}: คิว {queue_no} | วันที่ {txn.txn_date} | เวลา {timestamp_str} | คุณ {display_name} | {service} | ยอด {format_currency(amount)} | ยอดคงเหลือสุทธิ {format_currency(account.balance)}")

    return txn

In [80]:
# --- [ทดสอบเรียกใช้งานจริงกับลูกค้า 1 คน] ---
transaction_a = simulate_customer_visit(txn_id=1, customer_name="สมหญิง สายทอง", pause=0.5)

🎫 [ผู้ออกบัตรคิว] คุณ 'สมหญิง สายทอง' กดรับบัตรคิว ได้หมายเลข: A-001
🔔 [เชิญหมายเลข A-001] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: 'ถอนเงิน'
⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)
💻 เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:
หมายเลขคิว = 'A-001'
หมายเลขบัญชี = '430-1-33622-0'
ชื่อลูกค้า = 'สมหญิง สายทอง'
ประเภทรายการ = 'ถอนเงิน'
ยอดเงินก่อนทำรายการ = 2,776.46 บาท
จำนวนเงินทำรายการ = 529.01 บาท
ยอดเงินคงเหลือหลังทำรายการ = 2,281.16 บาท
สถานะรายการ = 'ถอนเงินสำเร็จ'
ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = 33.71 บาท
✅ ทำรายการสำเร็จ!
🧾 สลิปบันทึกรายการ #1: คิว A-001 | วันที่ 09/03/2026 | เวลา 09:29:22 | คุณ สมหญิง สายทอง | ถอนเงิน | ยอด 529.01 บาท | ยอดคงเหลือสุทธิ 2,281.16 บาท


##  ส่วนที่ 5 — จำลองลูกค้าหลายคนเดินเข้าธนาคารต่อเนื่องกัน


In [81]:
walk_in_customers = []

for i in range(2, 11):
    customer_name = fake.name()

    walk_in_customers.append(
        Member(
            member_id=i,
            customer_name=customer_name
        )
    )
completed_transactions = []  # เก็บผลลัพธ์ของทุกรายการในรอบนี้

for i, cust in enumerate(walk_in_customers, start=2):
    # ส่งชื่อลูกค้าเข้าฟังก์ชัน simulate_customer_visit
    txn = simulate_customer_visit(txn_id=i, customer_name=cust.customer_name, pause=0.3)
    completed_transactions.append(txn)

print("=" * 60)
print(f"🏁 จบการสาธิต — วันนี้มีลูกค้าเข้าทำรายการทั้งหมด {len(completed_transactions) + 1} คน (รวมคิว #1 ก่อนหน้า)")

🎫 [ผู้ออกบัตรคิว] คุณ 'ลัดดา เณรานุสนธิ์' กดรับบัตรคิว ได้หมายเลข: A-002
🔔 [เชิญหมายเลข A-002] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: 'ถอนเงิน'
⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)
💻 เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:
หมายเลขคิว = 'A-002'
หมายเลขบัญชี = '990-5-73908-0'
ชื่อลูกค้า = 'ลัดดา เณรานุสนธิ์'
ประเภทรายการ = 'ถอนเงิน'
ยอดเงินก่อนทำรายการ = 2,232.03 บาท
จำนวนเงินทำรายการ = 1,906.53 บาท
ยอดเงินคงเหลือหลังทำรายการ = 330.38 บาท
สถานะรายการ = 'ถอนเงินสำเร็จ'
ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = 4.88 บาท
✅ ทำรายการสำเร็จ!
🧾 สลิปบันทึกรายการ #2: คิว A-002 | วันที่ 17/03/2026 | เวลา 09:29:23 | คุณ ลัดดา เณรานุสนธิ์ | ถอนเงิน | ยอด 1,906.53 บาท | ยอดคงเหลือสุทธิ 330.38 บาท
🎫 [ผู้ออกบัตรคิว] คุณ 'ภัคชัญญา ทองแท้' กดรับบัตรคิว ได้หมายเลข: A-003
🔔 [เชิญหมายเลข A-003] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: 'ถอนเงิน'
⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)
💻 เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:
หมายเลขคิว = 'A-003'
หมายเลขบัญชี = '149-2-665

## ส่วนที่ 6 — สรุปผลจากการ demo (ยืนยันว่าฟังก์ชันคืนค่าถูกต้องและใช้ต่อได้จริง

In [88]:
import pandas as pd

# ดึงข้อมูลจาก transactions (300 รายการ) มาทำตาราง
summary_rows = [
    {
        "txn_id": t.txn_id,
        "queue_number": f"A-{t.txn_id:03d}",
        "txn.txn_date": t.txn_date,
        "account_number": t.account.account_number,
        "customer_name": " ".join(t.customer_name) if isinstance(t.customer_name, (tuple, list)) else t.customer_name,
        "transaction_type": t.transaction_type,
        "amount": t.amount,
        "balance_after": round(t.account.balance, 2),
        "target_account": t.target_account if t.target_account else "-"
    }
    for t in transactions  # 📌 ใช้ transactions แทน all_demo_transactions
]

summary_df = pd.DataFrame(summary_rows)
summary_df

,txn_id,queue_number,txn.txn_date,account_number,customer_name,transaction_type,amount,balance_after,target_account
0,1,A-001,16/06/2026,607-8-71898-0,พาสุข ถนัดหัตถกรรม,ฝากเงิน,355.29,2117.29,-
1,2,A-002,05/06/2026,955-7-66723-0,วิถี ดิสกะประกาย,ฝากเงิน,1598.57,4150.41,-
2,3,A-003,19/08/2026,838-4-87483-0,จินต์จุฑา ตั้งกุลงาม,โอนเงิน,1557.60,735.67,987-6-23399-0
3,4,A-004,25/02/2026,109-7-99978-0,ปัณณธร นิลวรรณ,ฝากเงิน,158.12,3492.51,-
4,5,A-005,30/03/2026,548-8-82464-0,นันทวุฒิ เดชคุ้ม,โอนเงิน,1479.12,1204.08,987-6-40550-0
...,...,...,...,...,...,...,...,...,...
295,296,A-296,23/04/2026,343-4-21413-0,ภาสวุฒิ เนื้อนุ่ม,ถอนเงิน,193.65,315.07,-
296,297,A-297,16/08/2026,688-1-38517-0,เลิศเดช ถาวรายุศม์,โอนเงิน,1346.38,437.97,987-6-51430-0
297,298,A-298,04/02/2026,611-9-55810-0,หรรษา ตั้งกุลงาม,ฝากเงิน,1148.99,5511.46,-
298,299,A-299,23/03/2026,680-5-89312-0,นิรุตต์ นาคะนคร,ถอนเงิน,229.78,3172.08,-


In [83]:
# Save to csv
summary_df.to_csv("สหกรณ์ออมทรัพย์.csv")

## ส่วนที่ 7 — สรุปผล 300 คน

In [84]:
# คำนวณยอดหมุนเวียนรวมทำรายการในรอบ demo
total_volume = sum(t.amount for t in transactions)
print(f"\n📊 ยอดธุรกรรมทำรายการรวมจากลูกค้าที่เดินเข้าธนาคารในรอบ demo นี้: {format_currency(total_volume)}")


📊 ยอดธุรกรรมทำรายการรวมจากลูกค้าที่เดินเข้าธนาคารในรอบ demo นี้: 315,092.25 บาท
